## MRI aging embedding extraction

Extract 512-dimensional gated MRI features using a trained RFG-MedicalNet model. Run cells in order after editing the paths. Python dependencies: torch, numpy, pandas, nibabel, tqdm.

This notebook constructs the ResNet-10 backbone from MedicalNet's ResNet source.

Inputs must already have the same preprocessing used during brain-age training. The extraction dataset only loads images, and it does not normalize, resize, skull-strip, or apply age bias correction. Image shape and finite voxel values are checked rather than silently altered.

## Imports and paths

In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd
import nibabel as nib
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

INPUT_CSV = Path("data/T1w_nifti_skullstrip.csv")
OUTPUT_CSV = Path("data/T1w_nifti_skullstrip_MRI_Embeddings.csv")
IMAGE_PATH_COLUMN = "image_path_hdbet"
MODEL_DEFINITION_PATH = Path("RFG_MedicalNet/model.py")
MEDICALNET_RESNET_PATH = Path("MedicalNet/models/resnet.py")
TRAINED_BRAIN_AGE_PATH = Path("weights/RFG_MedicalNet_best.pth")
TARGET_SHAPE = (224, 224, 224)
EMBEDDING_LAYER_NAME = "aging_gate"
EXPECTED_EMBEDDING_DIM = 512
BATCH_SIZE = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load the trained RFG-MedicalNet model

In [ ]:
# Load the brain-age architecture
spec = importlib.util.spec_from_file_location("brain_age_model_definition", MODEL_DEFINITION_PATH)
if spec is None or spec.loader is None:
    raise ImportError(f"Cannot load {MODEL_DEFINITION_PATH}")
brain_age_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(brain_age_module)

# Construct the backbone architecture
resnet_spec = importlib.util.spec_from_file_location("medicalnet_resnet", MEDICALNET_RESNET_PATH)
if resnet_spec is None or resnet_spec.loader is None:
    raise ImportError(f"Cannot load {MEDICALNET_RESNET_PATH}")
resnet_module = importlib.util.module_from_spec(resnet_spec)
resnet_spec.loader.exec_module(resnet_module)
base_model = resnet_module.resnet10(
    sample_input_W=224, sample_input_H=224, sample_input_D=224,
    shortcut_type="B", no_cuda=DEVICE.type != "cuda", num_seg_classes=1
)

# Use the original class name, or the name in the organized brain-age model.py.
RFG_MedicalNet = getattr(brain_age_module, "RFG_MedicalNet", None)
model = RFG_MedicalNet(base_model, feature_dim=512, dropout=0.3).to(DEVICE)
checkpoint = torch.load(TRAINED_BRAIN_AGE_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()
print("Loaded trained brain-age model on", DEVICE)


## Read the input CSV

`image_path_hdbet` is the path to the final preprocessed MRI (`.nii.gz`). Change IMAGE_PATH_COLUMN if your CSV uses another column. Retain participant identifiers and clinical variables in this CSV; they will be preserved in the output.

In [ ]:
mci_df = pd.read_csv(INPUT_CSV)
if IMAGE_PATH_COLUMN not in mci_df.columns:
    raise ValueError(f"Missing CSV column: {IMAGE_PATH_COLUMN}")
if mci_df.empty or mci_df[IMAGE_PATH_COLUMN].isna().any():
    raise ValueError("CSV must contain rows with nonmissing image paths.")
if "row_index" in mci_df.columns or any(c.startswith("emb_") for c in mci_df.columns):
    raise ValueError("Input already contains extraction output columns; use the original input CSV.")
print("Rows:", len(mci_df))


In [ ]:
class MRIDatasetNifti(Dataset):
    def __init__(self, df, image_path_col, target_shape=(224, 224, 224)):
        self.df = df.reset_index(drop=True)
        self.image_path_col = image_path_col
        self.target_shape = target_shape

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row[self.image_path_col]

        img = nib.load(img_path)
        volume = img.get_fdata()

        if volume.shape != self.target_shape:
            raise ValueError(f"{img_path}: expected {self.target_shape}, got {volume.shape}")
        if not np.isfinite(volume).all():
            raise ValueError(f"{img_path}: MRI contains NaN or infinite values")
        volume = volume.astype(np.float32)
        
        # Shape: D, H, W → C, D, H, W
        volume = volume[None, :, :, :]

        volume = torch.tensor(volume, dtype=torch.float32)

        return volume, idx

In [ ]:
dataset = MRIDatasetNifti(
    mci_df,
    image_path_col=IMAGE_PATH_COLUMN,
    target_shape=TARGET_SHAPE
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [ ]:
sample_img, sample_idx = next(iter(loader))

print("Sample image shape:", sample_img.shape)
print("Sample index:", sample_idx)

## Extract gated MRI features

In [ ]:
embedding_layer_name = EMBEDDING_LAYER_NAME
embedding_outputs = {}

def get_embedding_hook(name):
    def hook(module, input, output):
        # If output is tuple/list, take the first tensor
        if isinstance(output, (tuple, list)):
            output = output[0]
        embedding_outputs[name] = output.detach().cpu()
    return hook

modules = dict(model.named_modules())
if embedding_layer_name not in modules:
    raise ValueError(f"Model has no layer named {embedding_layer_name}")
target_layer = modules[embedding_layer_name]


In [ ]:
all_embeddings = []
all_pred_ages = []
all_indices = []

model.eval()

hook_handle = target_layer.register_forward_hook(get_embedding_hook(embedding_layer_name))
try:
    with torch.no_grad():
        for imgs, indices in tqdm(loader, total=len(loader)):
            imgs = imgs.to(DEVICE)
    
            embedding_outputs.clear()
    
            preds = model(imgs)
    
            # predicted brain age
            preds_np = preds.detach().cpu().numpy().reshape(-1)
            all_pred_ages.extend(preds_np)
    
            # embedding
            emb = embedding_outputs[embedding_layer_name]
    
            emb = emb.reshape(emb.size(0), -1).numpy()
            all_embeddings.append(emb)
    
            all_indices.extend(indices.numpy())
finally:
    hook_handle.remove()


all_embeddings = np.vstack(all_embeddings)

print("Embeddings shape:", all_embeddings.shape)
print("Predicted ages:", len(all_pred_ages))

## Save embeddings

`emb_1`–`emb_512`: gated MRI features in their extraction order. All original columns are retained.

In [ ]:
embedding_cols = [f"emb_{i+1}" for i in range(all_embeddings.shape[1])]

emb_df = pd.DataFrame(all_embeddings, columns=embedding_cols)
emb_df["row_index"] = all_indices

# Merge with original dataframe
mci_with_emb = mci_df.reset_index(drop=True).copy()
mci_with_emb["row_index"] = mci_with_emb.index

mci_with_emb = mci_with_emb.merge(emb_df, on="row_index", how="left", validate="one_to_one", sort=False)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
mci_with_emb.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)
print(mci_with_emb.shape)